In [ ]:
import pandas as pd

#randomly sample a part of the original dataset
#dataset: https://www.kaggle.com/datasets/robikscube/flight-delay-dataset-20182022?resource=download&select=readme.html


chunk_list = []
for chunk in pd.read_csv("Combined_Flights_2022.csv", chunksize=100000):
  chunk_sampled = chunk.sample(frac=0.15, random_state=42)
  chunk_list.append(chunk_sampled)

df = pd.concat(chunk_list, ignore_index=True)

df.to_csv('Flights_2022_600k.csv', index=False)

In [2]:
import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score


df = pd.read_csv("Flights_2022_600k.csv")

In [3]:
df = df[df["Cancelled"] == False]#only flights not cancelled

df["is_delayed"] = (df["ArrDelayMinutes"]> 15).astype(int)#1 if delayed, 0 if arrived on time

df["TotalTaxiTime"] = df["TaxiOut"] + df["TaxiIn"] #taxi queue time + taxi arrival in queue time, total ground congestion at both airports

congestion_time_interval = [df["TotalTaxiTime"] <= 25, (df["TotalTaxiTime"] >25) & (df["TotalTaxiTime"] <=45), df["TotalTaxiTime"] >45]
congestion_temp = ["Normal", "Moderate", "Severe"]
df["TaxiCongestionFlag"] = np.select(congestion_time_interval, congestion_temp, default ="Normal")

df["Depart_Hour"] = df["CRSDepTime"] // 100

df["IsWeekend"] = np.where(df["DayOfWeek"].isin([6,7]), 1, 0) # 1 if weekend, else 0


In [ ]:
df.to_csv("600k_flight_output.csv", index=False)

In [4]:
df = pd.get_dummies(df, columns=["TaxiCongestionFlag"], drop_first=True)
y = df["is_delayed"]


columns_to_exclude = ["is_delayed", "ArrDelay", "ArrDelayMinutes", "DepDelay", "DepDelayMinutes", "Cancelled", "Diverted", "FlightDate",
                   "Tail_Number", "Origin", "Dest", "Airline", "Operating_Airline", "OriginCityName", "DestCityName",
                   "Marketing_Airline_Network", "Operated_or_Branded_Code_Share_Partners", "IATA_Code_Marketing_Airline",
                   "IATA_Code_Operating_Airline", "Flight_Number_Marketing_Airline", "Flight_Number_Operating_Airline", "OriginState",
                   "OriginStateName", "DestState", "DestStateName", "DepTimeBlk", "ArrTimeBlk"];
leaked_features = ["ArrDel15", "ArrivalDelayGroups", "ArrDelay", "ArrDelayMinutes", "ArrTime", "CRSArrTime"]
columns_to_exclude.extend(leaked_features)


X = df.drop(columns = columns_to_exclude, errors= "ignore")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)


rf_model = RandomForestClassifier( n_estimators = 100, max_depth = 15, random_state = 42, n_jobs = -1)
rf_model.fit(X_train, y_train)


y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]


print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred))

ROC-AUC Score: 0.9866
              precision    recall  f1-score   support

           0       0.95      0.99      0.97     93884
           1       0.94      0.82      0.88     24745

    accuracy                           0.95    118629
   macro avg       0.94      0.90      0.92    118629
weighted avg       0.95      0.95      0.95    118629



In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=X_train.columns)
print(importance.nlargest(10))

DepartureDelayGroups         0.384226
DepDel15                     0.335948
TotalTaxiTime                0.054089
TaxiCongestionFlag_Severe    0.027110
TaxiOut                      0.025141
WheelsOn                     0.020011
CRSElapsedTime               0.016411
WheelsOff                    0.013919
DepTime                      0.013043
CRSDepTime                   0.013028
dtype: float64


In [ ]:

from sqlalchemy import create_engine
engine = create_engine("sqlite:///flight2_600k.db")
df.to_sql(name="flights", con=engine, if_exists="replace", index=False, chunksize=10000)

593145

In [ ]:
import sqlite3
import pandas as pd

# Connect to your SQLite database
conn = sqlite3.connect('flight2_600k.db') # Replace with your actual database filename

# Execute query and load into DataFrame
query = """
SELECT
    Origin,
    COUNT(*) AS Total_Flights,
    ROUND(AVG(TaxiOut), 2) AS Avg_Taxi_Out,
    ROUND(AVG(TaxiIn), 2) AS Avg_Taxi_In,
    ROUND(AVG(TaxiOut) - AVG(TaxiIn), 2) AS Taxi_Asymmetry_Gap
FROM flights
GROUP BY Origin
HAVING Total_Flights > 5000
ORDER BY Total_Flights DESC
LIMIT 25;
"""

temp_df = pd.read_sql(query, conn)
conn.close()

print(temp_df)

   Origin  Total_Flights  Avg_Taxi_Out  Avg_Taxi_In  Taxi_Asymmetry_Gap
0     ATL          26946         15.35         6.65                8.70
1     ORD          25153         21.07         6.93               14.15
2     DFW          23482         17.80         6.76               11.04
3     DEN          22999         17.54         6.51               11.03
4     CLT          18445         21.27         6.88               14.38
5     LAX          16569         16.31         7.76                8.55
6     SEA          14561         18.15         6.75               11.40
7     LAS          14323         16.23         7.44                8.79
8     PHX          14177         14.18         6.81                7.36
9     LGA          14074         24.79         8.00               16.80
10    IAH          12831         17.81         7.41               10.40
11    MCO          12400         18.14         8.36                9.78
12    DCA          12222         20.88         7.74             